# Classifying TNBC Molecular Subtypes with Routine Clinical Data and a Low-Cost IHC Panel

**Aja Starkey**

### Purpose
Triple-negative breast cancer (TNBC) is defined by what it lacks: estrogen receptor (ER) and progesterone receptor (PR) expression, and HER2 overexpression or amplification, as determined by IHC and FISH testing. That definition groups together tumors with very different biology. Lehmann et al. divided TNBC into four molecular subtypes (BL1, BL2, LAR, and M) using gene expression profiling, but that kind of testing is expensive and is not part of routine care for TNBC patients.

The goal of this project is to classify TNBC subtypes without tumor sequencing. I asked two questions:
1. Can routine clinical data (age, staging, histology, demographics), collected for every patient at diagnosis, predict TNBC subtype?
2. How much does adding a small panel of inexpensive IHC stains (AR, Ki-67, CK5/6, EGFR, vimentin, FOXA1) improve that prediction?

TCGA does not include results for these stains, so I used the tumors' mRNA levels for the matching genes as stand-ins. This gives an estimate of what a real IHC panel could achieve, which would need to be confirmed with actual staining data.

**Primary metric:** macro F1, which weights all four subtypes equally. Accuracy can look acceptable while a model ignores the smaller subtypes, and in a clinical setting both false positives and false negatives matter.

**How to run:** Runtime → Run all. The notebook downloads everything it needs (about 150 MB the first time) and takes roughly 5 minutes on a standard Colab runtime.

## 1. Setup

In [ ]:
import os, urllib.request, warnings
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_validate, cross_val_predict)
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
SUBTYPES = ["BL1", "BL2", "LAR", "M"]
print("Ready")

## 2. Data

| Data | Source |
|---|---|
| Clinical data | TCGA-BRCA PanCancer Atlas (Liu et al., 2018, *Cell*), via the cBioPortal data repository |
| RNA-seq (used only for the 7 IHC stand-in genes) | Same TCGA-BRCA PanCancer Atlas study |
| Subtype labels | Lehmann et al. (2016), *PLoS ONE* 11(6): e0157368, Supplementary Table S1, "Refined TNBCtype" column |

In [ ]:
BASE = ("https://media.githubusercontent.com/media/cBioPortal/datahub/master/"
        "public/brca_tcga_pan_can_atlas_2018/")
FILES = {
    "tcga_brca_clinical_patient.txt": BASE + "data_clinical_patient.txt",
    "tcga_brca_mrna_rsem.txt":        BASE + "data_mrna_seq_v2_rsem.txt",   # ~150 MB
}
for local, url in FILES.items():
    if not os.path.exists(local):
        print(f"Downloading {local} ...")
        urllib.request.urlretrieve(url, local)
print("TCGA files ready")

clinical_all = pd.read_csv("tcga_brca_clinical_patient.txt", sep="\t", comment="#").set_index("PATIENT_ID")
print(f"Clinical data: {clinical_all.shape[0]} BRCA patients x {clinical_all.shape[1]} variables")

### Subtype labels

The cell downloads the S1 table from PLOS. If the download fails, it asks you to upload the file (`journal.pone.0157368.s006.xlsx`, from the article's Supporting Information).

In [ ]:
S1_URL = "https://journals.plos.org/plosone/article/file?type=supplementary&id=10.1371/journal.pone.0157368.s006"
S1_FILE = "lehmann_2016_S1.xlsx"

if not os.path.exists(S1_FILE):
    try:
        r = requests.get(S1_URL, timeout=60)
        r.raise_for_status()
        open(S1_FILE, "wb").write(r.content)
        print("Downloaded Lehmann S1 table")
    except Exception as e:
        print(f"Download failed ({e}). Please upload the S1 table (.xlsx).")
        from google.colab import files
        up = files.upload()
        open(S1_FILE, "wb").write(next(iter(up.values())))

s1 = pd.read_excel(S1_FILE)
s1 = s1.iloc[1:]                                   # first row is the table title / header text
labels = pd.DataFrame({
    "TCGA_ID": s1.iloc[:, 0].astype(str).str[:12],  # TCGA barcode -> 12-character patient ID
    "Subtype": s1.iloc[:, 5].astype(str).str.strip()  # "Refined TNBCtype" column
})
labels = labels[labels["TCGA_ID"].str.startswith("TCGA") & labels["Subtype"].isin(SUBTYPES)]
labels = labels.drop_duplicates("TCGA_ID").set_index("TCGA_ID")
print(f"{len(labels)} labeled TNBC patients")
print(labels["Subtype"].value_counts())

## 3. Merge and describe the cohort

In [ ]:
cohort = labels.join(clinical_all, how="inner")
print(f"Patients with both subtype labels and clinical data: {len(cohort)}\n")
print(cohort["Subtype"].value_counts().reindex(SUBTYPES).to_string())

age = cohort["AGE"]
print(f"\nAge: median {age.median():.0f} (IQR {age.quantile(.25):.0f}-{age.quantile(.75):.0f}), "
      f"range {age.min():.0f}-{age.max():.0f}; {(age < 50).mean():.0%} under 50")
print("Median age by subtype:", cohort.groupby("Subtype")["AGE"].median().to_dict())

fig, ax = plt.subplots(figsize=(6, 3.5))
cohort["Subtype"].value_counts().reindex(SUBTYPES).plot.bar(ax=ax, color="#2f6fae")
ax.set_ylabel("Patients"); ax.set_title("TNBC subtype distribution (TCGA)"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## 4. Routine clinical features

I keep only information that is collected for every patient as part of a standard diagnosis and does not require molecular testing.

| Kept (9) | Removed, and why |
|---|---|
| Age | Survival / recurrence outcomes: these happen after diagnosis (leakage) |
| Overall AJCC stage | `BUFFA_HYPOXIA_SCORE`: calculated from tumor mRNA, so it requires gene-expression testing |
| T, N and M stage | `AJCC_STAGING_EDITION`: which edition of the staging manual was used, which reflects diagnosis year and hospital rather than tumor biology |
| Histology (ICD-O-3) | `RADIATION_THERAPY`: a treatment given after diagnosis |
| Race, ethnicity | Constant columns (sex, cancer type), IDs, administrative dates, PAM50 subtype |
| Prior cancer diagnosis | Lymph node presentation (35% missing), genetic ancestry (redundant with race) |

Race and ethnicity are social categories rather than tumor biology. I keep them because TNBC incidence differs across groups, but they should be interpreted with care.

In [ ]:
CLIN_NUM = ["AGE"]
CLIN_CAT = ["AJCC_PATHOLOGIC_TUMOR_STAGE", "PATH_T_STAGE", "PATH_N_STAGE", "PATH_M_STAGE",
            "ICD_O_3_HISTOLOGY", "RACE", "ETHNICITY", "PRIOR_DX"]

clinical = cohort[CLIN_NUM + CLIN_CAT].copy()
clinical[CLIN_NUM] = clinical[CLIN_NUM].apply(pd.to_numeric, errors="coerce")
clinical[CLIN_CAT] = clinical[CLIN_CAT].astype(object)

missing = clinical.isna().mean().mul(100).round(1)
print("Missing values (%):"); print(missing[missing > 0].to_string())
clinical.head()

## 5. Low-cost IHC panel (mRNA stand-ins)

| Stain | Gene used as stand-in | Why it's included |
|---|---|---|
| Androgen receptor | *AR* | Defines the LAR subtype |
| FOXA1 | *FOXA1* | Partner of AR in luminal/androgen signaling |
| Ki-67 | *MKI67* | Proliferation (BL1) |
| CK5/6 | *KRT5*, *KRT6A* | Basal/myoepithelial markers (BL2) |
| EGFR | *EGFR* | Growth factor signaling (BL2) |
| Vimentin | *VIM* | Mesenchymal marker (M) |

All six are established IHC stains that can be run on the diagnostic biopsy. Values are log2(RSEM + 1) mRNA expression.

**Limitation:** protein staining scored by a pathologist is less precise than mRNA, so these results estimate the best a real IHC panel could do.

In [ ]:
IHC_GENES = {"AR": "Androgen receptor", "FOXA1": "FOXA1", "MKI67": "Ki-67",
             "KRT5": "CK5/6", "KRT6A": "CK5/6", "EGFR": "EGFR", "VIM": "Vimentin"}

expr = pd.read_csv("tcga_brca_mrna_rsem.txt", sep="\t")
expr = expr[expr["Hugo_Symbol"].isin(IHC_GENES)].drop(columns="Entrez_Gene_Id")
expr = expr.groupby("Hugo_Symbol").mean()                    # one row per gene
expr = expr.loc[:, expr.columns.str.endswith("-01")]         # primary tumors only
expr.columns = expr.columns.str[:12]                         # patient-level ID
expr = expr.loc[:, ~expr.columns.duplicated()]

ihc = np.log2(expr.clip(lower=0) + 1).T
ihc.columns = [f"IHC_{g}" for g in ihc.columns]
IHC_COLS = list(ihc.columns)

data = clinical.join(ihc, how="inner").join(cohort["Subtype"])
y = data["Subtype"].values
print(f"Final modeling dataset: {data.shape[0]} patients, {len(CLIN_NUM + CLIN_CAT)} clinical features "
      f"+ {len(IHC_COLS)} IHC stand-in features")
data.to_csv("tnbc_clinical_ihc_dataset.csv")

### Biology check

If the labels and markers line up, LAR tumors should show high *AR* and *FOXA1*, M tumors high *VIM*, and BL2 tumors high *KRT5/KRT6A* and *EGFR*.

In [ ]:
long = data[IHC_COLS + ["Subtype"]].melt(id_vars="Subtype", var_name="Marker", value_name="log2 expression")
long["Marker"] = long["Marker"].str.replace("IHC_", "")
g = sns.catplot(data=long, x="Subtype", y="log2 expression", col="Marker", col_wrap=4,
                kind="box", order=SUBTYPES, sharey=False, height=2.6, color="#8fb3d9")
g.fig.suptitle("IHC panel markers by Lehmann subtype (mRNA stand-ins)", y=1.03)
plt.savefig("ihc_markers_by_subtype.png", dpi=150, bbox_inches="tight")
plt.show()
data.groupby("Subtype")[IHC_COLS].median().round(2)

## 6. Models: clinical only vs. IHC panel vs. both

- **Models:** logistic regression and random forest, both with class weighting for the imbalanced subtypes
- **Validation:** repeated stratified 5-fold cross-validation (5 repeats = 25 fits per model). With 175 patients, a single 80/20 split puts only about 35 patients in the test set, so results swing a lot depending on which patients land there.
- **No leakage:** imputation, scaling and one-hot encoding are fit inside each training fold only

In [ ]:
def preprocessor(num_cols, cat_cols):
    steps = [("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                               ("scale", StandardScaler())]), num_cols)]
    if cat_cols:
        steps.append(("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
                                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols))
    return ColumnTransformer(steps)

MODELS = {
    "Logistic Regression": lambda: LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000),
    "Random Forest":       lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                          class_weight="balanced",
                                                          random_state=RANDOM_STATE, n_jobs=-1),
}
FEATURE_SETS = {
    "Clinical only (9)":         (CLIN_NUM, CLIN_CAT),
    "IHC panel only (7)":        (IHC_COLS, []),
    "Clinical + IHC panel (16)": (CLIN_NUM + IHC_COLS, CLIN_CAT),
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
rows = []
for fs_name, (num, cat) in FEATURE_SETS.items():
    for m_name, make_model in MODELS.items():
        pipe = Pipeline([("prep", preprocessor(num, cat)), ("model", make_model())])
        s = cross_validate(pipe, data[num + cat], y, cv=cv, scoring=["f1_macro", "accuracy"])
        rows.append({"Features": fs_name, "Model": m_name,
                     "Macro F1": s["test_f1_macro"].mean(), "Macro F1 SD": s["test_f1_macro"].std(),
                     "Accuracy": s["test_accuracy"].mean()})
        print(f"{fs_name:28s} {m_name:20s} macro F1 = {rows[-1]['Macro F1']:.3f} ± {rows[-1]['Macro F1 SD']:.3f}")

results = pd.DataFrame(rows)
results.to_csv("model_comparison.csv", index=False)
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
order = list(FEATURE_SETS)
colors = {"Logistic Regression": "#2f6fae", "Random Forest": "#8fb3d9"}
w = 0.38
for i, m_name in enumerate(MODELS):
    r = results[results["Model"] == m_name].set_index("Features").loc[order]
    x = np.arange(len(order)) + (i - 0.5) * w
    ax.bar(x, r["Macro F1"], w, yerr=r["Macro F1 SD"], capsize=3, color=colors[m_name], label=m_name)
ax.axhline(0.25, ls=":", c="grey"); ax.text(2.45, 0.26, "chance (0.25)", color="grey", fontsize=9, ha="right")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
ax.set_ylabel("Macro F1 (mean ± SD, 25 CV folds)"); ax.set_ylim(0, 1)
ax.set_title("Adding a low-cost IHC panel to routine clinical features")
ax.legend(loc="upper left")
plt.tight_layout(); plt.savefig("model_comparison.png", dpi=150); plt.show()

## 7. Per-subtype performance

Out-of-fold predictions from one round of 5-fold CV (random forest). Each patient is predicted by a model that never saw them during training.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, fs_name in zip(axes, ["Clinical only (9)", "Clinical + IHC panel (16)"]):
    num, cat = FEATURE_SETS[fs_name]
    pipe = Pipeline([("prep", preprocessor(num, cat)), ("model", MODELS["Random Forest"]())])
    pred = cross_val_predict(pipe, data[num + cat], y, cv=cv5)
    print(f"\n{fs_name}")
    print(classification_report(y, pred, labels=SUBTYPES, digits=2))
    cm = confusion_matrix(y, pred, labels=SUBTYPES, normalize="true")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=SUBTYPES, yticklabels=SUBTYPES, ax=ax)
    ax.set_title(fs_name); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=150); plt.show()

## 8. Which stains are worth their cost?

I remove one stain at a time and measure how much macro F1 drops. A small drop can mean a stain is unimportant *or* that another stain carries the same information. For example, AR and FOXA1 both flag LAR tumors.

In [ ]:
STAINS = {"AR": ["IHC_AR"], "Ki-67": ["IHC_MKI67"], "CK5/6": ["IHC_KRT5", "IHC_KRT6A"],
          "EGFR": ["IHC_EGFR"], "Vimentin": ["IHC_VIM"], "FOXA1": ["IHC_FOXA1"]}
cv_drop = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

def score(cols):
    pipe = Pipeline([("prep", preprocessor(cols, [])), ("model", MODELS["Random Forest"]())])
    return cross_validate(pipe, data[cols], y, cv=cv_drop, scoring="f1_macro")["test_score"].mean()

full = score(IHC_COLS)
drop_df = pd.DataFrame([{"Stain removed": s, "Macro F1 without it": score([c for c in IHC_COLS if c not in cols])}
                        for s, cols in STAINS.items()])
drop_df["Drop in macro F1"] = full - drop_df["Macro F1 without it"]
drop_df = drop_df.sort_values("Drop in macro F1", ascending=False).reset_index(drop=True)
print(f"Full 6-stain panel: macro F1 = {full:.3f}\n")
drop_df.round(3)

## 9. Results and Conclusions

*(These numbers are from my run. Small differences can appear across library versions.)*

| Features | Macro F1 (mean of 25 CV folds) |
|---|---|
| Routine clinical only (9 features) | 0.25–0.27, about chance level (0.25) |
| IHC panel only (6 stains) | 0.58–0.61 |
| Clinical + IHC panel | 0.58–0.59 |

**1. Routine clinical data cannot identify TNBC subtypes.** With only routinely available features, the model performs at chance. An earlier version of this project reached a macro F1 of 0.36, but that relied mostly on the Buffa hypoxia score, which is calculated from tumor mRNA and so requires gene-expression testing.

**2. A small IHC panel recovers much of the subtype signal.** Six stains roughly doubled macro F1 compared with clinical data alone. LAR (F1 ≈ 0.75) and BL1 (≈ 0.68) were identified best; BL2 and M remained difficult to separate. Adding the clinical features to the panel did not improve it further.

**3. Vimentin, CK5/6 and Ki-67 contributed the most** in the drop-one analysis. AR and FOXA1 each looked less important only because they overlap: both mark LAR tumors.

**4. Cost.** A six-stain IHC panel costs roughly $450–1,000 depending on payer (CPT 88342/88341), and it runs on biopsy tissue already collected for diagnosis. For comparison, commercial gene-expression tests such as Oncotype DX average about $2,300–5,300 in commercial reimbursement. These are estimates from published fee data, not quotes.

### Limitations
- The IHC features are mRNA stand-ins. Real stains are scored on coarser scales (0–3+, % positive), so real-world performance would likely be somewhat lower.
- 175 patients from a single cohort (TCGA), with no external validation.
- The Lehmann labels were derived from gene expression, which gives any expression-based feature a built-in advantage.
- Insurers generally cover IHC stains when the results guide treatment. Routine subtype-based staining will likely depend on subtypes becoming linked to specific therapies, such as AR-targeted drugs for LAR tumors, which are currently being studied in clinical trials.

### Future Work
- Validate with real IHC data. Zhao et al. (*The Oncologist*, 2020) showed that a related TNBC subtyping scheme can be assigned with four IHC stains (AR, CD8, FOXC1, DCLK1).
- Add tumor-infiltrating lymphocyte (TIL) scores from H&E slides, which every patient already has.
- Test on an independent cohort such as METABRIC.